[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/RO/Seminars/seminar1_randamente_stationaritate.ipynb)

# Seminar 1: Randamente și Staționaritate

## Statistica Piețelor Financiare
**Academia de Studii Economice din București**

Prof. dr. Daniel Traian Pele | Antoaneta Amza

---

### Obiective
1. Calculați randamente simple și logaritmice din date de preț
2. Comparați agregarea temporală a celor două tipuri de randamente
3. Explicați pierderea din varianță (variance drag) și impactul ei
4. Argumentați de ce analizăm randamente și nu prețuri (staționaritate)
5. Implementați calculul randamentelor în Python

In [ ]:
# Importuri necesare
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from scipy.stats import jarque_bera
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.graphics.tsaplots import plot_acf
import warnings
warnings.filterwarnings('ignore')

# Stil grafice - Nature journal quality
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 0.75

# Culori
MAIN_BLUE = '#1A3A6E'
CRIMSON   = '#DC3545'
FOREST    = '#2E7D32'
AMBER     = '#B5853F'
ORANGE    = '#E67E22'

print('Importuri complete!')

## Partea I: Calcul manual — Randamente simple și logaritmice

### Exercițiul 1

Date de preț: $P_0 = 100$, $P_1 = 110$, $P_2 = 99$, $P_3 = 120$

**Randament simplu:** $R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$

**Randament logaritmic:** $r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$

In [ ]:
# Exercițiul 1: Calcul manual
prices = np.array([100, 110, 99, 120])
t = np.arange(len(prices))

# Randamente simple
R = np.diff(prices) / prices[:-1]

# Randamente logaritmice
r = np.log(prices[1:] / prices[:-1])

# Tabel rezultate
df_returns = pd.DataFrame({
    'Perioadă': ['t=0→1', 't=1→2', 't=2→3'],
    'P_{t-1}': prices[:-1],
    'P_t': prices[1:],
    'R_t (simplu)': [f'{x:.4f} ({x*100:.2f}%)' for x in R],
    'r_t (log)': [f'{x:.4f} ({x*100:.3f}%)' for x in r]
})
print(df_returns.to_string(index=False))

print(f"\nSuma R_1 + R_2 = {R[0] + R[1]:.4f} (dar prețul a scăzut la 99!)")
print(f"Suma r_1 + r_2 + r_3 = {r.sum():.5f}")
print(f"ln(P_3/P_0) = ln(120/100) = {np.log(120/100):.5f} ✓")

### Observație cheie

- $R_1 + R_2 = 0\%$, dar prețul a scăzut la 99! Randamentele simple **nu sunt aditive** — se compun multiplicativ: $\prod(1+R_t)$
- Log-randamentele **se adună**: $r_1 + r_2 + r_3 = \ln(P_3/P_0)$ ✓
- Folosim log-randamente în modelarea statistică pentru **aditivitate temporală**

In [ ]:
# Verificare: compunere multiplicativă
compound_simple = np.prod(1 + R)
compound_log = np.exp(np.sum(r))

print(f"Compunere multiplicativă: ∏(1+R_t) = {compound_simple:.4f}")
print(f"Compunere logaritmică:    exp(Σr_t) = {compound_log:.4f}")
print(f"Randament total simplu:   {(compound_simple - 1)*100:.2f}%")
print(f"Randament total log:      {np.sum(r)*100:.3f}%")
print(f"\nAmbele metode → aceeași avere finală: {prices[0] * compound_simple:.0f} EUR")

## Partea II: Variance Drag

### Sondaj rapid

Investiți **$100** într-un activ. Acesta crește cu +50% într-un an, apoi scade cu -50% în anul următor. Cu cât rămâneți?

**Răspuns:** $100 × 1.50 × 0.50 = **$75** (pierdere de 25%, nu 0%!)

### Formula Variance Drag

$$\bar{r}_{\text{geometric}} \approx \bar{r}_{\text{aritmetic}} - \frac{\sigma^2}{2}$$

Volatilitatea **distruge** averea pe termen lung.

In [ ]:
# Demonstrație Variance Drag
print("=" * 50)
print("VARIANCE DRAG: Demonstrație")
print("=" * 50)

# Exemplul +50%, -50%
initial = 100
after_up = initial * 1.50
after_down = after_up * 0.50
print(f"\n$100 × 1.50 = ${after_up:.0f}")
print(f"${after_up:.0f} × 0.50 = ${after_down:.0f}")
print(f"Pierdere: {(after_down/initial - 1)*100:.0f}%")

# Formula: pierdere ≈ σ²/2
sigma = 0.50  # 50% volatilitate
drag = sigma**2 / 2
print(f"\nVariance drag: σ²/2 = {sigma}²/2 = {drag:.4f} = {drag*100:.1f}%")
print(f"Media aritmetică a randamentelor: (+50% + -50%)/2 = 0%")
print(f"Media geometrică: 0% - {drag*100:.1f}% = -{drag*100:.1f}%")
print(f"Verificare: √(1.50 × 0.50) - 1 = {(np.sqrt(1.50*0.50) - 1)*100:.2f}%")

In [ ]:
# Exercițiul 2: Variance drag pe diferite niveluri de volatilitate
print("\nImpactul variance drag pe orizonturi diferite:")
print(f"{'σ anual':>10} {'Drag/an':>10} {'Drag 10 ani':>12} {'Drag 30 ani':>12}")
print("-" * 50)

for sigma in [0.10, 0.16, 0.30, 0.50, 0.75]:
    drag_annual = sigma**2 / 2
    drag_10y = 1 - (1 - drag_annual)**10
    drag_30y = 1 - (1 - drag_annual)**30
    print(f"{sigma*100:>8.0f}%  {drag_annual*100:>8.1f}%  {drag_10y*100:>10.1f}%  {drag_30y*100:>10.1f}%")

print("\nExemple: S&P 500 (σ≈16%), Acțiuni individuale (σ≈30-40%), Bitcoin (σ≈75%)")

In [ ]:
# Vizualizare Variance Drag
fig, ax = plt.subplots(figsize=(7, 4))

sigmas = np.linspace(0.01, 1.0, 200)
drag_pct = (sigmas**2 / 2) * 100

ax.plot(sigmas * 100, drag_pct, color=MAIN_BLUE, linewidth=1.5)
ax.fill_between(sigmas * 100, 0, drag_pct, alpha=0.1, color=MAIN_BLUE)

# Marcăm active specifice
assets = {'S&P 500': 16, 'Acțiuni': 35, 'Bitcoin': 75}
colors_assets = [FOREST, AMBER, CRIMSON]
for (name, vol), c in zip(assets.items(), colors_assets):
    drag_val = (vol/100)**2 / 2 * 100
    ax.plot(vol, drag_val, 'o', color=c, markersize=6, zorder=5)
    ax.annotate(f'{name}\n({drag_val:.1f}%/an)', xy=(vol, drag_val),
                xytext=(vol+5, drag_val+3), fontsize=7,
                arrowprops=dict(arrowstyle='->', color=c, lw=0.7))

ax.set_xlabel('Volatilitate anuală (%)')
ax.set_ylabel('Variance drag (%/an)')
ax.set_title('Variance Drag: $\\sigma^2/2$', fontweight='bold')
ax.set_xlim(0, 100)
ax.set_ylim(0, 55)

plt.tight_layout()
plt.show()

## Partea III: Staționaritate

### De ce analizăm randamente, nu prețuri?

O serie temporală $\{X_t\}$ este **strict staționară** dacă distribuția $(X_{t_1}, ..., X_{t_k})$ nu depinde de $t$.

O serie este **slab staționară** dacă:
- $\mathbb{E}[X_t] = \mu$ (constantă)
- $\text{Var}(X_t) = \sigma^2$ (constantă)
- $\text{Cov}(X_t, X_{t+h})$ depinde doar de $h$, nu de $t$

**Prețurile** au trend → nestaționare → metodele statistice clasice nu sunt valide

**Randamentele** sunt (aproximativ) staționare → putem aplica teste statistice

## Partea IV: Implementare Python — Descărcare date și calcul randamente

In [ ]:
# Descărcare date S&P 500
print("Descărcare date S&P 500...")
sp500 = yf.download('^GSPC', start='2000-01-01', end='2025-12-31', progress=False)
close = sp500['Close'].squeeze()

# Calculul randamentelor
log_ret = np.log(close / close.shift(1)).dropna()
simple_ret = close.pct_change().dropna()

print(f"\nPerioada: {close.index[0].strftime('%Y-%m-%d')} — {close.index[-1].strftime('%Y-%m-%d')}")
print(f"Observații: {len(log_ret)}")
print(f"\n--- Statistici descriptive (log-randamente) ---")
print(f"Media zilnică:      {log_ret.mean():.6f}")
print(f"Media anualizată:   {log_ret.mean() * 252:.4f} ({log_ret.mean() * 252 * 100:.2f}%)")
print(f"Volatilitate zilnică: {log_ret.std():.6f}")
print(f"Vol. anualizată:    {log_ret.std() * np.sqrt(252):.4f} ({log_ret.std() * np.sqrt(252) * 100:.2f}%)")
print(f"Skewness:           {log_ret.skew():.4f}")
print(f"Kurtosis (excess):  {log_ret.kurtosis():.4f}")
print(f"Minim:              {log_ret.min():.4f} ({log_ret.min() * 100:.2f}%)")
print(f"Maxim:              {log_ret.max():.4f} ({log_ret.max() * 100:.2f}%)")

In [ ]:
# Grafic: Preț vs. Randamente
fig, axes = plt.subplots(2, 1, figsize=(12, 6), gridspec_kw={'height_ratios': [1.2, 1]})

# Panoul superior: Prețul de închidere
axes[0].plot(close.index, close.values, color=MAIN_BLUE, linewidth=0.6)
axes[0].set_title('S&P 500: Preț de închidere', fontweight='bold')
axes[0].set_ylabel('Preț ($)')

# Panoul inferior: Log-randamente
axes[1].plot(log_ret.index, log_ret.values, color=MAIN_BLUE, linewidth=0.4, alpha=0.7)
axes[1].axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
axes[1].set_title('S&P 500: Log-randamente zilnice', fontweight='bold')
axes[1].set_ylabel('Log-randament')

plt.tight_layout()
plt.show()

print("Observați: Prețul are trend (nestaționaritate), randamentele oscilează în jurul lui zero.")
print("Volatility clustering evident în perioadele de criză (2008, 2020).")

In [ ]:
# Testul ADF (Augmented Dickey-Fuller)
print("=" * 60)
print("TESTUL ADF (Augmented Dickey-Fuller)")
print("=" * 60)
print("H0: Seria are rădăcină unitară (nestaționară)")
print("H1: Seria este staționară")
print()

# ADF pe log-prețuri
log_price = np.log(close.dropna())
adf_price = adfuller(log_price, autolag='AIC')
print(f"--- Log-prețuri ---")
print(f"  ADF Statistic: {adf_price[0]:.4f}")
print(f"  p-value:       {adf_price[1]:.4f}")
print(f"  Lags:          {adf_price[2]}")
print(f"  Valori critice: 1%: {adf_price[4]['1%']:.3f}, 5%: {adf_price[4]['5%']:.3f}")
print(f"  Concluzie:     {'Staționară ✓' if adf_price[1] < 0.05 else 'NESTAȚIONARĂ ✗ (nu respingem H0)'}")

print()

# ADF pe log-randamente
adf_ret = adfuller(log_ret, autolag='AIC')
print(f"--- Log-randamente ---")
print(f"  ADF Statistic: {adf_ret[0]:.4f}")
print(f"  p-value:       {adf_ret[1]:.6f}")
print(f"  Lags:          {adf_ret[2]}")
print(f"  Valori critice: 1%: {adf_ret[4]['1%']:.3f}, 5%: {adf_ret[4]['5%']:.3f}")
print(f"  Concluzie:     {'Staționară ✓' if adf_ret[1] < 0.05 else 'NESTAȚIONARĂ ✗'}")

In [ ]:
# Testul Jarque-Bera (normalitate)
print("=" * 60)
print("TESTUL JARQUE-BERA (normalitate)")
print("=" * 60)
print("H0: Datele sunt distribuite normal")
print("H1: Datele NU sunt distribuite normal")
print()

jb_stat, jb_p = jarque_bera(log_ret)
print(f"JB Statistic: {jb_stat:.1f}")
print(f"p-value:      {jb_p:.2e}")
print(f"Skewness:     {log_ret.skew():.4f} (Normal: 0)")
print(f"Kurtosis:     {log_ret.kurtosis() + 3:.4f} (Normal: 3)")
print(f"\nConcluzie: {'NU sunt normale ✗' if jb_p < 0.05 else 'Nu respingem normalitatea'} (p={jb_p:.2e})")
print("\nRandamentele financiare nu sunt normale: cozi groase + leptokurtoză")

In [ ]:
# Histogramă + distribuția normală suprapusă
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panoul stâng: histogramă normală
axes[0].hist(log_ret, bins=100, density=True, alpha=0.5,
             color=MAIN_BLUE, edgecolor='white', linewidth=0.3,
             label='S&P 500 log-randamente')
x_range = np.linspace(log_ret.min() * 1.2, log_ret.max() * 1.2, 500)
normal_pdf = stats.norm.pdf(x_range, loc=log_ret.mean(), scale=log_ret.std())
axes[0].plot(x_range, normal_pdf, color=CRIMSON, linewidth=1.5,
             linestyle='--', label='Normal')
axes[0].set_title('Distribuția empirică vs. Normală', fontweight='bold')
axes[0].set_xlabel('Log-randament')
axes[0].set_ylabel('Densitate')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
               ncol=2, frameon=False)

# Panoul drept: QQ-plot
stats.probplot(log_ret, dist='norm', plot=axes[1])
axes[1].get_lines()[0].set(color=MAIN_BLUE, markersize=2, alpha=0.5)
axes[1].get_lines()[1].set(color=CRIMSON, linewidth=1.5)
axes[1].set_title('QQ-Plot vs. Normală', fontweight='bold')

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

print("Observații:")
print("- Vârful central mai ascuțit decât Normala (leptokurtoză)")
print("- Cozile mai groase (fat tails) → evenimente extreme mai frecvente")
print("- QQ-plot: formă de S → devierea de la normalitate în cozi")

In [ ]:
# ACF: Randamente vs. Randamente pătratice
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ACF pe randamente
acf_vals = acf(log_ret, nlags=30)
axes[0].bar(range(31), acf_vals, width=0.4, color=MAIN_BLUE, alpha=0.7)
axes[0].axhline(y=1.96/np.sqrt(len(log_ret)), color=CRIMSON, linestyle='--', linewidth=0.8)
axes[0].axhline(y=-1.96/np.sqrt(len(log_ret)), color=CRIMSON, linestyle='--', linewidth=0.8)
axes[0].set_title('ACF: $r_t$ (randamente)', fontweight='bold')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('Autocorelație')

# ACF pe randamente pătratice
acf_sq = acf(log_ret**2, nlags=30)
axes[1].bar(range(31), acf_sq, width=0.4, color=CRIMSON, alpha=0.7)
axes[1].axhline(y=1.96/np.sqrt(len(log_ret)), color=MAIN_BLUE, linestyle='--', linewidth=0.8)
axes[1].axhline(y=-1.96/np.sqrt(len(log_ret)), color=MAIN_BLUE, linestyle='--', linewidth=0.8)
axes[1].set_title('ACF: $r_t^2$ (randamente pătratice)', fontweight='bold')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('Autocorelație')

plt.tight_layout()
plt.show()

print("Concluzii:")
print("- ACF(r_t) ≈ 0: randamentele NU sunt autocorelate (consistent cu EMH)")
print("- ACF(r_t²) ≠ 0: VOLATILITY CLUSTERING - perioadele volatile persistă")
print("- Randamentele sunt necorelate dar NU independente")
print("- Aceasta este motivația modelelor GARCH")

In [ ]:
# Scatter plot: Randamente simple vs. logaritmice
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(simple_ret.values, log_ret.values, s=2, alpha=0.3, color=MAIN_BLUE)
ax.plot([-0.15, 0.15], [-0.15, 0.15], color=CRIMSON, linewidth=1, linestyle='--', label='Linia 45°')

# Corelație
corr = np.corrcoef(simple_ret.values, log_ret.values)[0, 1]
ax.text(0.05, 0.95, f'Corelație: {corr:.6f}', transform=ax.transAxes,
        fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Randament simplu ($R_t$)')
ax.set_ylabel('Randament logaritmic ($r_t$)')
ax.set_title('Randamente simple vs. logaritmice', fontweight='bold')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), frameon=False)
ax.set_xlim(-0.15, 0.15)
ax.set_ylim(-0.15, 0.15)

plt.tight_layout()
plt.show()

print(f"Corelația dintre R_t și r_t: {corr:.6f}")
print("Pentru valori mici, R_t ≈ r_t (aproximarea Taylor de ordinul I)")

## Studiu de caz: Comparație pe multiple active

In [ ]:
# Comparație statistici pe multiple active
tickers = {
    'S&P 500': '^GSPC',
    'Bitcoin': 'BTC-USD',
    'EUR/USD': 'EURUSD=X',
    'Gold': 'GC=F',
    'AAPL': 'AAPL'
}

results = []
for name, ticker in tickers.items():
    try:
        data = yf.download(ticker, start='2015-01-01', end='2025-12-31', progress=False)
        c = data['Close'].squeeze().dropna()
        r = np.log(c / c.shift(1)).dropna()
        r = r[np.isfinite(r)]
        
        adf = adfuller(r, autolag='AIC')
        jb = jarque_bera(r)
        
        results.append({
            'Activ': name,
            'N': len(r),
            'Medie (%)': f'{r.mean()*100:.4f}',
            'Vol. anuală (%)': f'{r.std()*np.sqrt(252)*100:.1f}',
            'Skewness': f'{r.skew():.3f}',
            'Kurtosis': f'{r.kurtosis()+3:.1f}',
            'ADF p-val': f'{adf[1]:.4f}',
            'JB p-val': f'{jb[1]:.2e}'
        })
    except Exception as e:
        print(f"{name}: Eroare - {e}")

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
print("\nToate activele: ADF respinge H0 (staționaritate) și JB respinge normalitatea")

In [ ]:
# Tabel momentele empirice vs. cele ale distribuției normale
print("\n" + "=" * 60)
print("COMPARAȚIE: Momente empirice vs. Normală")
print("=" * 60)
print(f"\n{'Moment':<20} {'S&P 500':>12} {'Normală':>12}")
print("-" * 45)
print(f"{'Skewness':<20} {log_ret.skew():>12.4f} {'0.0000':>12}")
print(f"{'Kurtosis':<20} {log_ret.kurtosis()+3:>12.4f} {'3.0000':>12}")
print(f"{'Min':<20} {log_ret.min():>12.4f} {stats.norm.ppf(1/len(log_ret), scale=log_ret.std()):>12.4f}")
print(f"{'Max':<20} {log_ret.max():>12.4f} {stats.norm.ppf(1-1/len(log_ret), scale=log_ret.std()):>12.4f}")

# Evenimente extreme
sigma = log_ret.std()
print(f"\n{'Eveniment extreme':<30} {'Observat':>10} {'Așteptat (Normal)':>18}")
print("-" * 60)
for k in [3, 4, 5]:
    obs = (np.abs(log_ret) > k * sigma).sum()
    exp = len(log_ret) * 2 * stats.norm.sf(k)
    print(f"{'|r| > ' + str(k) + 'σ':<30} {obs:>10d} {exp:>18.1f}")

print("\nConcluzii: evenimentele extreme sunt MULT mai frecvente decât predicția Gaussiană!")

## Temă pentru acasă

1. Alegeți o acțiune listată la BVB (ex: TLV.RO, SNP.RO, BRD.RO)
2. Descărcați datele zilnice din ultimii 5 ani
3. Calculați log-randamentele
4. Aplicați testul ADF (pe prețuri și pe randamente)
5. Aplicați testul Jarque-Bera
6. Realizați graficele: preț vs. randamente, histogramă, QQ-plot, ACF
7. Comparați cu S&P 500: sunt randamentele BVB „mai Gaussiene”?

---

## Rezumat

| Concept | Formulă / Test | Concluzie |
|---------|---------------|----------|
| Log-randamente | $r_t = \ln(P_t/P_{t-1})$ | Aditive temporal |
| Variance drag | $\approx \sigma^2/2$ | Volatilitatea distruge averea |
| Staționaritate | Test ADF | Prețuri: I(1), Randamente: I(0) |
| Normalitate | Test Jarque-Bera | Respinsă: cozi groase, leptokurtoză |
| Volatility clustering | ACF($r_t^2$) | Perioadele volatile persistă |